# 2.1 理论计算题

**输入：** 彩色图像大小 $3 \times 32 \times 32$（通道数 × 高 × 宽）

**卷积层：** 16 个卷积核，每个大小 $3 \times 5 \times 5$，填充（Padding）= 2，步幅（Stride）= 2

### （1）输出特征图尺寸

输出高度：

$$
H_{out}
=
\left\lfloor
\frac{H+2p-k}{s}
\right\rfloor+1
=
\left\lfloor
\frac{32+2\times2-5}{2}
\right\rfloor+1
=
\left\lfloor
\frac{31}{2}
\right\rfloor+1
=
15+1
=
16
$$

输出宽度同理为 16，输出通道数等于卷积核个数 16。

**答案：**

$$
16 \times 16 \times 16
$$

### （2）单个输出通道的一个像素值的乘法次数

每个卷积核包含：

$$
3 \times 5 \times 5 = 75
$$

个权重参数。

生成一个输出像素值需要进行 75 次乘法。

**答案：**

$$
75
$$

## 2.2 编程题 – 手动实现最大池化前向传播

In [7]:
import numpy as np

def max_pool2d_forward(x, kernel_size, stride, padding=0):
    """
    二维最大池化前向传播（支持 stride 和 padding）
    x: 输入张量，形状 (N, C, H, W) 或 (C, H, W)
    kernel_size: int 或 (kh, kw)
    stride: int 或 (sh, sw)
    padding: int 或 (ph, pw)
    """
    # 参数标准化
    if isinstance(kernel_size, int):
        k_h = k_w = kernel_size
    else:
        k_h, k_w = kernel_size
    if isinstance(stride, int):
        s_h = s_w = stride
    else:
        s_h, s_w = stride
    if isinstance(padding, int):
        p_h = p_w = padding
    else:
        p_h, p_w = padding

    # 统一为 4D 张量
    if x.ndim == 3:
        x = x[np.newaxis, ...]
        squeezed = True
    else:
        squeezed = False
    N, C, H, W = x.shape

    # 填充
    x_pad = np.pad(x, ((0,0), (0,0), (p_h, p_h), (p_w, p_w)),mode='constant', constant_values=-np.inf)

    # 输出尺寸
    out_h = (H + 2*p_h - k_h) // s_h + 1
    out_w = (W + 2*p_w - k_w) // s_w + 1
    out = np.zeros((N, C, out_h, out_w))

    # 滑动窗口取最大值
    for i in range(out_h):
        for j in range(out_w):
            h_start = i * s_h
            h_end = h_start + k_h
            w_start = j * s_w
            w_end = w_start + k_w
            window = x_pad[:, :, h_start:h_end, w_start:w_end]
            out[:, :, i, j] = np.max(window, axis=(2, 3))

    return out[0] if squeezed else out

# 示例
if __name__ == "__main__":
    x = np.random.randn(1, 3, 32, 32)
    out = max_pool2d_forward(x, kernel_size=2, stride=2, padding=0)
    print("最大池化输出形状:", out.shape)

最大池化输出形状: (1, 3, 16, 16)


# 3.1 理论计算题

VGG 中使用多个 $3\times3$ 卷积级联代替大卷积核，输入和输出特征图通道数均为 $C$，且不带偏置。

### （1）一个 $5\times5$ 卷积层的参数量

$$
5 \times 5 \times C \times C
=
25C^2
$$

**答案：**

$$
25C^2
$$

### （2）两个串联的 $3\times3$ 卷积层总参数量

单个卷积层参数量：

$$
3 \times 3 \times C \times C
=
9C^2
$$

两个卷积层总参数量：

$$
2 \times 9C^2
=
18C^2
$$

**答案：**

$$
18C^2
$$

## 3.2 编程题 – NiN 块定义

In [ ]:
import torch.nn as nn

class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(NiNBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU()
        )
    def forward(self, x):
        return self.conv(x)

# 示例：输入通道3，输出通道16，3x3卷积，步幅1，填充1
block = NiNBlock(3, 16, kernel_size=3, stride=1, padding=1)
print(block)

NiNBlock(
  (conv): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
    (3): ReLU()
    (4): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
    (5): ReLU()
  )
)


# 4.1 理论计算题

给定：

$$
x_1=2,\quad x_2=4,\quad x_3=6,\quad x_4=8
$$

批量归一化参数：

$$
\gamma=2,\quad \beta=1,\quad \epsilon=0
$$

### （1）计算均值

$$
\mu
=
\frac{2+4+6+8}{4}
=
5
$$

### （2）计算方差

$$
\sigma^2
=
\frac{(2-5)^2+(4-5)^2+(6-5)^2+(8-5)^2}{4}
=
\frac{9+1+1+9}{4}
=
5
$$

标准差：

$$
\sigma=\sqrt{5}
$$

### （3）批量归一化

公式：

$$
y
=
\gamma\frac{x-\mu}{\sigma}
+
\beta
$$

计算：

$$
y_1
=
2\cdot\frac{2-5}{\sqrt5}+1
=
-\frac{6}{\sqrt5}+1
\approx -1.6833
$$

$$
y_2
=
2\cdot\frac{4-5}{\sqrt5}+1
=
-\frac{2}{\sqrt5}+1
\approx 0.1056
$$

$$
y_3
=
2\cdot\frac{6-5}{\sqrt5}+1
=
\frac{2}{\sqrt5}+1
\approx 1.8944
$$

$$
y_4
=
2\cdot\frac{8-5}{\sqrt5}+1
=
\frac{6}{\sqrt5}+1
\approx 3.6833
$$

**答案：**

$$
y_1\approx-1.6833,\quad
y_2\approx0.1056,\quad
y_3\approx1.8944,\quad
y_4\approx3.6833
$$


## 4.2 编程题 – 残差块 Residual

In [ ]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super(Residual, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        if use_1x1conv:
            self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        shortcut = self.shortcut(x)
        out += shortcut
        out = self.relu(out)
        return out

# 示例
res_block = Residual(3, 16, use_1x1conv=True, stride=2)
print(res_block)

Residual(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (shortcut): Conv2d(3, 16, kernel_size=(1, 1), stride=(2, 2))
)


# 5.1 理论计算题

### （1）为什么底层特征提取层设小学习率（或冻结），顶层输出层设大学习率？

底层特征（边缘、纹理、形状等）已经在大型数据集（如 ImageNet）上学习到了通用表示，具有较强的迁移能力。

若目标数据集较小且与源数据集相似，应尽量保留这些通用特征，因此采用较小学习率或直接冻结参数，避免破坏已有知识。

顶层分类器需要适应新的类别划分，通常随机初始化，因此需要较大的学习率以加快收敛速度。

### （2）目标数据集非常小且与源数据集非常相似时的微调策略

- 冻结大部分底层卷积层；
- 仅微调最后几层或重新训练分类头；
- 使用较小学习率（如预训练阶段的 1/10）；
- 使用数据增强（随机裁剪、翻转、颜色扰动等）；
- 使用 Dropout、标签平滑等正则化方法抑制过拟合。


## 5.2 编程题 – 图像增广管道

In [ ]:
from torchvision import transforms

augmentation_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),  # 随机裁剪并缩放到224x224
    transforms.RandomHorizontalFlip(p=0.5),               # 50%概率水平翻转
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),  # 颜色抖动
    transforms.ToTensor()                                 # 转换为张量
])

print(augmentation_pipeline)

Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
    ToTensor()
)


# 6.1 理论计算题

真实框：

$$
A=[10,10,50,50]
$$

预测框：

$$
B=[30,30,70,70]
$$

### （1）交集面积

交集左上角：

$$
(30,30)
$$

交集右下角：

$$
(50,50)
$$

交集面积：

$$
(50-30)\times(50-30)
=
20\times20
=
400
$$

### （2）并集面积

真实框面积：

$$
(50-10)\times(50-10)
=
40\times40
=
1600
$$

预测框面积：

$$
(70-30)\times(70-30)
=
40\times40
=
1600
$$

并集面积：

$$
1600+1600-400
=
2800
$$

### （3）IoU

$$
IoU
=
\frac{400}{2800}
=
\frac{1}{7}
\approx 0.142857
$$

**答案：**

$$
IoU
=
\frac{1}{7}
\approx 0.142857
$$

## 6.2 编程题 – 标签平滑交叉熵损失

In [ ]:
import torch
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, labels, epsilon=0.1, reduction='mean'):
    """
    logits: (N, K) 未归一化的分数
    labels: (N,) 真实类别索引
    epsilon: 平滑因子
    reduction: 'mean', 'sum', 'none'
    """
    K = logits.size(-1)
    with torch.no_grad():
        smooth_targets = torch.full_like(logits, epsilon / (K - 1))
        smooth_targets.scatter_(1, labels.unsqueeze(1), 1 - epsilon)
    log_probs = F.log_softmax(logits, dim=-1)
    loss = - (smooth_targets * log_probs).sum(dim=-1)
    if reduction == 'mean':
        return loss.mean()
    elif reduction == 'sum':
        return loss.sum()
    else:
        return loss

# 示例
logits = torch.randn(4, 10)
labels = torch.tensor([0, 1, 2, 3])
loss = label_smoothing_cross_entropy(logits, labels, epsilon=0.1)
print("标签平滑交叉熵损失:", loss.item())

标签平滑交叉熵损失: 2.5002059936523438
